# TMA lymphocyte panel — cell type proportions

**Goal:** Compare cell type abundances between MHC II–positive and MHC II–negative
patients in TMA cores stained with the lymphocyte panel (Panel 3: DAPI, CD3, CD56,
CD8, MHC I, MHC II, PanCK). Cell counts are normalized by total cells per core
(tumor + stroma combined) and compared by Mann-Whitney U test with
Benjamini-Hochberg FDR correction.

**Input:** Panel 3 TMA summary CSV (one row per core, HALO export).

**Output:** Figure 5A (tumor PanCK+ proportions), supplementary figure
(PanCK- T cell proportions).

In [ ]:
from pathlib import Path
import yaml
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out  = fig_dir   / 'figure5'
supp_out = fig_dir   / 'figure5-supp'
table_out = table_dir / 'figure5'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

custom_palette = {
    'class II positive': cmap_high_low[0],
    'class II negative': cmap_high_low[1],
}

## Load data

Panel 3 TMA summary CSV and patient MHC II classifications. Region and patient
identifiers are extracted from the HALO image tag strings, which encode TMA number,
row letter, and core position. Tonsil cores are used as internal controls and
excluded from the patient-level analysis.

In [ ]:
#panel3_path        = Path('/project/Dolatshahi_Lab/Gabe_Hanson/data/mhc2_vectra_data/object_data/tmas/panel3/panel3_tma_object_data_6_19_25.csv')
panel3_path        = Path('/project/Dolatshahi_Lab/Gabe_Hanson/data/mhc2_vectra_data/object_data/tmas/panel3/panel3_tma_summary_data_6_19_25.csv')
patient_class_path = Path(cfg['datasets']['patient_classifications']['raw'])

panel3 = pd.read_csv(panel3_path)
patient_classifications = pd.read_csv(patient_class_path, index_col=0)
patient_classifications.loc['tonsil'] = ['Tonsil', 'Tonsil', 'Tonsil']
patient_classifications['patient'] = patient_classifications.index

panel3["tma"] = panel3["Image Tag"].str.extract(r"LUNGADCA2014-(\d+)").astype(float).astype("Int64")

print(f"Panel 3 cores loaded: {len(panel3):,}")

## Extract region and patient identifiers

Each TMA core image tag encodes the TMA number, row letter, and core position.
Region labels (N = normal adjacent, PT = primary tumor, CT = core tumor) are
mapped from core position indices. Patient IDs are derived from TMA number and
row letter using a fixed layout. Tonsil controls are labeled separately.

In [ ]:
# extract region
# special-case tonsil cores — hardcoded positions confirmed from TMA layout
TONSIL_CORES = [
    "TMA3 T8-LUNGADCA2014-1_Core[1,A,6]", "TMA3 T8-LUNGADCA2014-1_Core[1,A,7]",
    "TMA3 T8-LUNGADCA2014-3_Core[1,4,A]", "TMA3 T8-LUNGADCA2014-3_Core[1,7,A]",
    "TMA3 T8-LUNGADCA2014-2_Core[1,A,7]", "TMA3 T8-LUNGADCA2014-4_Core[1,A,1]",
    "TMA3 T8-LUNGADCA2014-4_Core[1,A,5]"
]

REGION_MAP = {1: 'N', 5: 'N', 2: 'PT', 6: 'PT', 3: 'CT', 7: 'CT'}

def extract_region(image_tag):
    if any(t in image_tag for t in TONSIL_CORES):
        return 'tonsil'
    match = re.search(r'Core\[\d+,[A-Z],(\d+)\]', image_tag)
    if not match:
        match = re.search(r'Core\[\d+,(\d+),[A-Z]\]', image_tag)
    if not match:
        return ''
    return REGION_MAP.get(int(match.group(1)), '')

panel3['region'] = panel3['Image Tag'].apply(extract_region)

In [ ]:
#extract patient IDs
TMA_BASE = {
    1: (1,  14),
    2: (27, 40),
    3: (53, 66),
    4: (79, 92),
}

def extract_patient(image_tag, region):
    if region == 'tonsil':
        return 'tonsil'
    tma_match = re.search(r'LUNGADCA2014-(\d+)', image_tag)
    if not tma_match:
        return ''
    tma_number = int(tma_match.group(1))

    match1 = re.search(r'Core\[\d+,([A-Z]),(\d+)\]', image_tag)
    match2 = re.search(r'Core\[\d+,(\d+),([A-Z])\]', image_tag)
    if match1:
        row_letter, last_number = match1.groups()
    elif match2:
        last_number, row_letter = match2.groups()
    else:
        return ''

    row_index  = ord(row_letter) - ord('B')
    last_number = int(last_number)

    if tma_number not in TMA_BASE:
        return ''

    base_5_6_7, base_1_2_3 = TMA_BASE[tma_number]

    if tma_number == 4:
        if last_number == 1:
            base = 92
        elif last_number in [3, 4, 5]:
            base = 79
        else:
            return ''
    else:
        if last_number in [1, 2, 3]:
            base = base_1_2_3
        elif last_number in [5, 6, 7]:
            base = base_5_6_7
        else:
            return ''

    return base + row_index

panel3['patient'] = panel3.apply(
    lambda x: extract_patient(x['Image Tag'], x['region']), axis=1
)

In [ ]:
# merge classifications
panel3 = pd.merge(panel3, patient_classifications, on='patient', how='left')
panel3 = panel3.loc[~panel3['patient classification'].isna()].copy()
panel3['patient_region'] = panel3['patient'].astype(str) + panel3['region'].astype(str)

print(panel3['patient classification'].value_counts())
print(f"Cores after removing tonsil and unclassified: {len(panel3):,}")

## Derive combined cell type columns

Composite cell type columns are computed by summing HALO phenotype subsets.
MHCII+/- totals collapse MHC I co-expression status. CD3+ totals combine
CD8+ and CD8- subsets. Total (tumor + stroma) columns are added for NK and
T cell comparisons across compartments.

In [ ]:
# MHC II collapsed totals
panel3['tumor: PanCK+MHCII+ Cells'] = (
    panel3['tumor: PanCK+MHCI+MHCII+ Cells'] +
    panel3['tumor: PanCK+MHCI-MHCII+ Cells']
)
panel3['tumor: PanCK+MHCII- Cells'] = (
    panel3['tumor: PanCK+MHCI+MHCII- Cells'] +
    panel3['tumor: PanCK+MHCI-MHCII- Cells']
)

# CD3+ totals per compartment
panel3['stroma: PanCK-CD3+ Cells'] = (
    panel3['stroma: PanCK-CD3+CD8+ Cells'] +
    panel3['stroma: PanCK-CD3+CD8- Cells']
)
panel3['tumor: PanCK-CD3+ Cells'] = (
    panel3['tumor: PanCK-CD3+CD8+ Cells'] +
    panel3['tumor: PanCK-CD3+CD8- Cells']
)

# Cross-compartment totals
panel3['total PanCK-CD3+ Cells'] = (
    panel3['tumor: PanCK-CD3+ Cells'] +
    panel3['stroma: PanCK-CD3+ Cells']
)
panel3['total PanCK-CD3-CD56+ Cells'] = (
    panel3['tumor: PanCK-CD3-CD56+ Cells'] +
    panel3['stroma: PanCK-CD3-CD56+ Cells']
)
panel3['total Total Cells'] = (
    panel3['tumor: Total Cells'] +
    panel3['stroma: Total Cells']
)

## Aggregate by patient_region and run statistics

Cell counts are summed across cores sharing the same patient and region, then
normalized by total cells (tumor + stroma) per patient_region. Mann-Whitney U
tests compare MHC II–positive vs negative groups for each cell type, with
Benjamini-Hochberg FDR correction applied across all comparisons.

In [ ]:
# --- Figure 5A cell types: NK cells and T cells ---
cell_cols_fig5a = [
    'stroma: PanCK-CD3-CD56+ Cells',
    'stroma: PanCK-CD3+ Cells',
    'tumor: PanCK-CD3+ Cells',
    'total PanCK-CD3-CD56+ Cells',
    'total PanCK-CD3+ Cells',
]

# --- Supplemental cell types: MHC I/II quadrants + collapsed totals ---
cell_cols_supp = [
    'tumor: PanCK+MHCI+MHCII- Cells',
    'tumor: PanCK+MHCI-MHCII+ Cells',
    'tumor: PanCK+MHCI+MHCII+ Cells',
    'tumor: PanCK+MHCI-MHCII- Cells',
    'tumor: PanCK+MHCII- Cells',
    'tumor: PanCK+MHCII+ Cells',
]

# filter to classified patients only
panel3_classified = panel3.loc[
    panel3['patient classification'].isin(['class II positive', 'class II negative'])
].copy()

# aggregate by patient_region
all_cols = cell_cols_fig5a + cell_cols_supp
region_counts = (
    panel3_classified
    .groupby(['patient_region', 'patient classification'])[all_cols + ['total Total Cells']]
    .sum()
    .reset_index()
)

# normalize by total cells per core
for col in all_cols:
    region_counts[col] = region_counts[col] / region_counts['total Total Cells']

def run_stats(region_counts, cell_cols):
    """
    Run Mann-Whitney U tests with Benjamini-Hochberg FDR correction
    for a given set of cell type columns.

    Parameters
    ----------
    region_counts : pd.DataFrame
        Aggregated normalized counts with 'patient classification' column.
    cell_cols : list of str
        Cell type columns to test.

    Returns
    -------
    stats_df : pd.DataFrame
        Results with raw and FDR-adjusted p-values.
    pval_map : dict
        FDR-adjusted p-values keyed by cell type column name.
    """
    rows = []
    for col in cell_cols:
        pos = region_counts.loc[
            region_counts['patient classification'] == 'class II positive', col
        ].dropna()
        neg = region_counts.loc[
            region_counts['patient classification'] == 'class II negative', col
        ].dropna()
        if len(pos) > 0 and len(neg) > 0:
            _, p = mannwhitneyu(pos, neg, alternative='two-sided')
            rows.append({
                'Cell Type': col,
                'P-value': p,
                'Median (pos)': pos.median(),
                'Median (neg)': neg.median(),
                'Higher group': 'class II positive' if pos.median() > neg.median() else 'class II negative',
            })

    stats_df = pd.DataFrame(rows)
    _, fdr, _, _ = multipletests(stats_df['P-value'].values, method='fdr_bh')
    stats_df['FDR-adjusted P-value'] = fdr
    stats_df['Significant (FDR < 0.05)'] = fdr < 0.05
    stats_df = stats_df.sort_values('FDR-adjusted P-value')
    pval_map = stats_df.set_index('Cell Type')['FDR-adjusted P-value'].to_dict()
    return stats_df, pval_map

# run corrections independently for each figure
stats_fig5a, pval_map_fig5a = run_stats(region_counts, cell_cols_fig5a)
stats_supp,  pval_map_supp  = run_stats(region_counts, cell_cols_supp)

# save stats tables
stats_fig5a.to_csv(table_out / 'tma_panel3_lymphocyte_stats.csv', index=False)
stats_supp.to_csv(table_out  / 'tma_panel3_mhc_validation_stats.csv', index=False)

print("=== Figure 5A — lymphocytes ===")
print(stats_fig5a[['Cell Type', 'P-value', 'FDR-adjusted P-value', 'Significant (FDR < 0.05)']].to_string(index=False))
print("\n=== Supplemental — MHC I/II validation ===")
print(stats_supp[['Cell Type', 'P-value', 'FDR-adjusted P-value', 'Significant (FDR < 0.05)']].to_string(index=False))

## Figure 5A — TMA lymphocyte panel immune cell proportions

PanCK- immune cell type proportions (normalized by total cells per core)
compared between MHC II–positive and MHC II–negative patients using
`ceiba.plot_utils.make_comparison_figure`. Stars reflect FDR-adjusted
p-values from the shared stats run above.

In [ ]:
from ceiba.plot_utils import make_comparison_figure

# make_comparison_figure expects '{cell_type}_fraction' columns
# rename normalized cols to match that interface
plot_df = region_counts.copy()
for col in cell_cols_fig5a + cell_cols_supp:
    plot_df[f'{col}_fraction'] = plot_df[col]

In [ ]:
# Figure 5A
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_fig5a,
    neg_label='class II negative',
    pos_label='class II positive',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_fig5a),
    panel_size=(3.0, 5),
    pval_map=pval_map_fig5a,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
    outpath=fig_out / 'fig5a_tma_panel3_immune_proportions.pdf',
)
plt.show()

In [ ]:
# check counts per group
print("N patient_regions per classification:")
print(region_counts['patient classification'].value_counts())

# check raw vs normalized values for key columns
for col in cell_cols_fig5a:
    pos = region_counts.loc[region_counts['patient classification'] == 'class II positive', col]
    neg = region_counts.loc[region_counts['patient classification'] == 'class II negative', col]
    print(f"\n{col}")
    print(f"  pos: n={len(pos)}, median={pos.median():.4f}, zeros={( pos==0).sum()}")
    print(f"  neg: n={len(neg)}, median={neg.median():.4f}, zeros={(neg==0).sum()}")

# also check what the raw (unnormalized) tumor CD3 counts look like
raw_tumor_cd3 = panel3_classified.groupby(['patient_region', 'patient classification'])[
    ['tumor: PanCK-CD3+ Cells', 'tumor: Total Cells', 'stroma: Total Cells']
].sum().reset_index()
print("\nRaw tumor CD3+ counts sample:")
print(raw_tumor_cd3.head(20).to_string())

## Supplementary — tumor PanCK+ MHC I/II proportions

Tumor epithelial cell subsets stratified by MHC I and MHC II co-expression,
normalized by total cells per core.

In [ ]:
# Supplemental
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_supp,
    neg_label='class II negative',
    pos_label='class II positive',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_supp),
    panel_size=(3.0, 5),
    pval_map=pval_map_supp,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
    outpath=supp_out / 'figS5a_tma_panel3_mhc_validation.pdf',
)
plt.show()